# Analyse exploratoire — Télémétrie des nœuds (Enact)

Ce notebook analyse deux jeux de données de télémétrie collectés sur deux nœuds :
- `rpi4` : un Raspberry Pi 4 (device edge, ressources limitées)
- `vm-node` : une machine virtuelle (nœud cloud/fog)

Deux campagnes de mesure distinctes :
- **pods_on** : mesures prises pendant que des pods (workloads Kubernetes) sont déployés/actifs sur les nœuds
- **pods_off** : mesures prises à vide, sans pods déployés (baseline "idle")

Objectif : nettoyer les données (preprocessing) puis calculer des statistiques descriptives pour comprendre
la distribution de la consommation de ressources (CPU, mémoire, énergie, réseau) par nœud et par condition (on/off).

In [1]:
# --- Imports ---
# pandas : manipulation des données tabulaires
# numpy  : calculs numériques (utilisé pour les stats comme le coefficient de variation)
import pandas as pd
import numpy as np

# Options d'affichage pandas : on augmente le nombre de colonnes affichées
# et on force un affichage plus lisible des flottants (2 décimales) pour les stats descriptives.
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. Chargement des données

On charge les deux fichiers CSV séparément (ils correspondent à deux campagnes de mesure
différentes, pas à un seul dataset découpé). On ajoute une colonne `workload_status`
pour garder la trace de l'origine de chaque ligne ("on" ou "off"), ce qui permettra
de les combiner plus tard sans perdre l'information.

In [2]:
DATA_DIR = "dataset/enact"

# Chargement des deux campagnes de mesure.
df_on = pd.read_csv(f"{DATA_DIR}/node_telemetry_pods_on.csv")
df_off = pd.read_csv(f"{DATA_DIR}/node_telemetry_pods_off.csv")

# On étiquette chaque ligne avec sa condition d'origine ("on" = pods actifs, "off" = idle).
# C'est important pour ne pas perdre cette information une fois les deux dataframes fusionnés.
df_on["workload_status"] = "on"
df_off["workload_status"] = "off"

print(f"df_on  : {df_on.shape[0]} lignes, {df_on.shape[1]} colonnes")
print(f"df_off : {df_off.shape[0]} lignes, {df_off.shape[1]} colonnes")

df_on  : 18144 lignes, 9 colonnes
df_off : 18074 lignes, 9 colonnes


## 2. Inspection initiale

Avant de nettoyer quoi que ce soit, on regarde à quoi ressemblent les données brutes :
- les types de colonnes (`dtypes`) pour vérifier si pandas a bien deviné les types numériques
- un aperçu des premières lignes (`head`)
- un résumé technique (`info`) : types + nombre de valeurs non-nulles par colonne (repère rapide de valeurs manquantes)

In [3]:
# Aperçu des premières lignes du dataset "pods_on".
df_on.head()

,node_name,timestamp,CPU (%),MEM (%),fs (%),Energy (watts),rx (B/sec),tx (B/sec),workload_status
0,vm-node,2026-01-24 01:50:03,0.1878,0.5352,0.7597,83.7420,585737.6000,745078.7333,on
1,rpi4,2026-01-24 01:50:03,0.1082,0.7730,0.6920,83.5639,46397.8333,51134.8000,on
2,rpi4,2026-01-24 01:50:33,0.1052,0.7733,0.6920,83.2242,38643.0333,44372.4667,on
3,vm-node,2026-01-24 01:50:33,0.2405,0.5350,0.7597,85.1713,869018.6667,1112669.0000,on
4,rpi4,2026-01-24 01:51:03,0.1023,0.7723,0.6920,83.0453,38505.6333,43729.7667,on


In [4]:
# info() donne : le type de chaque colonne + le nombre de valeurs non-nulles.
# Utile pour repérer d'un coup d'œil si une colonne a des valeurs manquantes
# (si "non-null count" < nombre total de lignes) ou un type inattendu
# (ex: une colonne numérique lue comme "object" à cause d'une valeur texte parasite).
print("--- df_on ---")
df_on.info()
print("\n--- df_off ---")
df_off.info()

--- df_on ---
<class 'pandas.DataFrame'>
RangeIndex: 18144 entries, 0 to 18143
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   node_name        18144 non-null  str    
 1   timestamp        18144 non-null  str    
 2   CPU (%)          18144 non-null  float64
 3   MEM (%)          18144 non-null  float64
 4   fs (%)           18144 non-null  float64
 5   Energy (watts)   18144 non-null  float64
 6   rx (B/sec)       18144 non-null  float64
 7   tx (B/sec)       18144 non-null  float64
 8   workload_status  18144 non-null  str    
dtypes: float64(6), str(3)
memory usage: 1.2 MB

--- df_off ---
<class 'pandas.DataFrame'>
RangeIndex: 18074 entries, 0 to 18073
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   node_name        18074 non-null  str    
 1   timestamp        18074 non-null  str    
 2   CPU (%)          18074 non-null  

## 3. Nettoyage des noms de colonnes

Les noms de colonnes bruts contiennent des unités entre parenthèses et des espaces
(ex: `CPU (%)`, `Energy (watts)`), ce qui est peu pratique pour écrire du code
(impossible d'utiliser `df.cpu_pct` en notation pointée, et les espaces obligent
à passer par `df["CPU (%)"]`). On les renomme en `snake_case` sans unité,
en gardant l'unité en commentaire pour ne pas la perdre.

In [5]:
# Mapping ancien nom -> nouveau nom.
# Unités conservées ici en commentaire pour référence :
#   cpu_pct    : fraction d'utilisation CPU (0-1, PAS un pourcentage 0-100 malgré le nom "CPU (%)" d'origine)
#   mem_pct    : fraction d'utilisation mémoire (0-1)
#   fs_pct     : fraction d'utilisation du système de fichiers/disque (0-1)
#   energy_w   : consommation énergétique instantanée, en watts
#   rx_bps     : débit réseau entrant, en octets/seconde
#   tx_bps     : débit réseau sortant, en octets/seconde
rename_map = {
    "CPU (%)": "cpu_pct",
    "MEM (%)": "mem_pct",
    "fs (%)": "fs_pct",
    "Energy (watts)": "energy_w",
    "rx (B/sec)": "rx_bps",
    "tx (B/sec)": "tx_bps",
}

df_on = df_on.rename(columns=rename_map)
df_off = df_off.rename(columns=rename_map)

print(df_on.columns.tolist())

['node_name', 'timestamp', 'cpu_pct', 'mem_pct', 'fs_pct', 'energy_w', 'rx_bps', 'tx_bps', 'workload_status']


## 4. Détection et suppression des doublons

On vérifie s'il existe des lignes strictement identiques (même nœud, même timestamp,
mêmes mesures). Un doublon signalerait un bug de collecte (même échantillon inséré
deux fois), pas une vraie observation supplémentaire — il faut donc le supprimer
pour ne pas biaiser les statistiques (une valeur comptée deux fois pèse artificiellement
plus dans la moyenne).

In [6]:
# duplicated() renvoie True pour chaque ligne qui est une copie exacte d'une ligne précédente.
# .sum() compte simplement combien de lignes sont marquées comme doublons.
n_dup_on = df_on.duplicated().sum()
n_dup_off = df_off.duplicated().sum()
print(f"Doublons dans df_on  : {n_dup_on}")
print(f"Doublons dans df_off : {n_dup_off}")

# On supprime les doublons s'il y en a (garde la première occurrence par défaut).
df_on = df_on.drop_duplicates()
df_off = df_off.drop_duplicates()

Doublons dans df_on  : 498
Doublons dans df_off : 282


## 5. Valeurs manquantes

On vérifie s'il y a des cellules vides (`NaN`). Pour de la télémétrie de capteur continue,
on n'en attend normalement pas, mais il faut vérifier plutôt que supposer :
un capteur peut avoir un raté de lecture ponctuel.

In [7]:
# isna() marque chaque cellule vide (True/False), .sum() compte par colonne le nombre de NaN.
print("--- Valeurs manquantes par colonne (df_on) ---")
print(df_on.isna().sum())
print("\n--- Valeurs manquantes par colonne (df_off) ---")
print(df_off.isna().sum())

# S'il y en avait, on choisirait ici entre dropna() (supprimer la ligne)
# ou fillna() (interpolation/valeur par défaut) selon leur proportion.
# Ici, on s'attend à 0 — la cellule ci-dessus sert de vérification, pas de correction.

--- Valeurs manquantes par colonne (df_on) ---
node_name          0
timestamp          0
cpu_pct            0
mem_pct            0
fs_pct             0
energy_w           0
rx_bps             0
tx_bps             0
workload_status    0
dtype: int64

--- Valeurs manquantes par colonne (df_off) ---
node_name          0
timestamp          0
cpu_pct            0
mem_pct            0
fs_pct             0
energy_w           0
rx_bps             0
tx_bps             0
workload_status    0
dtype: int64


## 6. Conversion des types (timestamp)

La colonne `timestamp` est actuellement lue comme du texte (`object`) par pandas.
On la convertit en véritable type datetime avec `pd.to_datetime`, ce qui permettra :
- de trier chronologiquement
- de calculer des écarts de temps entre mesures consécutives (étape 8)
- d'extraire des composantes (heure, jour) si besoin plus tard

In [8]:
df_on["timestamp"] = pd.to_datetime(df_on["timestamp"])
df_off["timestamp"] = pd.to_datetime(df_off["timestamp"])

# node_name et workload_status sont des catégories à faible cardinalité (2 valeurs chacune) :
# le type "category" est plus léger en mémoire et signale explicitement que ce sont
# des variables discrètes plutôt que du texte libre.
df_on["node_name"] = df_on["node_name"].astype("category")
df_off["node_name"] = df_off["node_name"].astype("category")

# On trie par nœud puis par timestamp : indispensable pour l'étape suivante
# (calcul des écarts de temps entre mesures consécutives), qui n'a de sens
# que si les lignes sont dans l'ordre chronologique au sein de chaque nœud.
df_on = df_on.sort_values(["node_name", "timestamp"]).reset_index(drop=True)
df_off = df_off.sort_values(["node_name", "timestamp"]).reset_index(drop=True)

print(df_on.dtypes)

node_name                category
timestamp          datetime64[us]
cpu_pct                   float64
mem_pct                   float64
fs_pct                    float64
energy_w                  float64
rx_bps                    float64
tx_bps                    float64
workload_status               str
dtype: object


## 7. Cohérence des plages de valeurs (sanity check)

On vérifie que chaque colonne numérique reste dans une plage physiquement plausible :
- `cpu_pct`, `mem_pct`, `fs_pct` : doivent rester entre 0 et 1 (c'est une fraction, pas un pourcentage 0-100)
- `energy_w`, `rx_bps`, `tx_bps` : doivent être positifs (une puissance ou un débit négatif n'a pas de sens physique)

Si une valeur sort de ces bornes, c'est un signal d'erreur de capteur ou de mauvaise unité,
pas une "vraie" valeur extrême à garder telle quelle.

In [9]:
def check_ranges(df, label):
    """Vérifie les bornes physiques attendues et affiche les violations éventuelles."""
    issues = {}
    for col in ["cpu_pct", "mem_pct", "fs_pct"]:
        out_of_range = df[(df[col] < 0) | (df[col] > 1)]
        if len(out_of_range) > 0:
            issues[col] = len(out_of_range)
    for col in ["energy_w", "rx_bps", "tx_bps"]:
        negative = df[df[col] < 0]
        if len(negative) > 0:
            issues[col] = len(negative)

    if issues:
        print(f"[{label}] Violations de plage détectées : {issues}")
    else:
        print(f"[{label}] Aucune violation de plage détectée — toutes les valeurs sont plausibles.")

check_ranges(df_on, "df_on")
check_ranges(df_off, "df_off")

[df_on] Aucune violation de plage détectée — toutes les valeurs sont plausibles.
[df_off] Aucune violation de plage détectée — toutes les valeurs sont plausibles.


## 8. Régularité temporelle de l'échantillonnage

D'après l'aperçu initial, les mesures semblent prises toutes les ~30 secondes par nœud.
On vérifie cette hypothèse en calculant l'écart de temps (`diff`) entre deux mesures
consécutives **au sein du même nœud** (`groupby("node_name")`, sinon on comparerait
des timestamps de nœuds différents entre eux, ce qui n'a pas de sens).

Un grand écart signalerait un trou dans la collecte (panne de capteur ou de réseau) —
important à savoir avant toute analyse de tendance temporelle.

In [10]:
def check_time_gaps(df, label, expected_seconds=30):
    """Calcule l'écart de temps entre mesures consécutives, par nœud, et repère les anomalies."""
    # groupby("node_name")["timestamp"].diff() : calcule (timestamp[i] - timestamp[i-1])
    # séparément pour chaque groupe de node_name, donc sans mélanger les deux nœuds.
    gaps = df.groupby("node_name")["timestamp"].diff().dt.total_seconds()

    print(f"--- {label} ---")
    print(f"Écart médian entre mesures : {gaps.median():.1f} s (attendu ~{expected_seconds} s)")
    print(f"Écart max observé          : {gaps.max():.1f} s")
    # On compte les écarts nettement supérieurs à l'attendu (ex: > 3x l'intervalle normal)
    # comme des "trous" probables dans la collecte.
    n_gaps = (gaps > expected_seconds * 3).sum()
    print(f"Nombre de trous suspects (> {expected_seconds * 3}s) : {n_gaps}")
    print()

check_time_gaps(df_on, "df_on")
check_time_gaps(df_off, "df_off")

--- df_on ---
Écart médian entre mesures : 30.0 s (attendu ~30 s)
Écart max observé          : 30.0 s
Nombre de trous suspects (> 90s) : 0

--- df_off ---
Écart médian entre mesures : 30.0 s (attendu ~30 s)
Écart max observé          : 603.0 s
Nombre de trous suspects (> 90s) : 5



## 9. Fusion des deux datasets pour l'analyse comparative

Maintenant que chaque dataset est nettoyé et étiqueté (`workload_status`), on les concatène
en un seul dataframe. Ça ne mélange pas l'information — grâce à `workload_status` et `node_name`,
on peut toujours regrouper et comparer séparément par la suite (`groupby`).

In [11]:
df_all = pd.concat([df_on, df_off], ignore_index=True)
df_all["workload_status"] = df_all["workload_status"].astype("category")

metric_cols = ["cpu_pct", "mem_pct", "fs_pct", "energy_w", "rx_bps", "tx_bps"]

print(f"Dataset fusionné : {df_all.shape[0]} lignes")
df_all.groupby(["node_name", "workload_status"], observed=True).size()

Dataset fusionné : 35438 lignes


node_name  workload_status
rpi4       off                8882
           on                 8823
vm-node    off                8910
           on                 8823
dtype: int64

## 10. Statistiques descriptives

**Important : on ne calcule jamais les stats sur l'ensemble mélangé.** `rpi4` et `vm-node`
ont des profils matériels totalement différents (un Raspberry Pi n'a rien à voir avec une VM
en termes de capacité CPU/énergie) — les moyenner ensemble n'aurait aucun sens physique.
On regroupe donc systématiquement par `node_name` × `workload_status` (4 combinaisons :
rpi4/on, rpi4/off, vm-node/on, vm-node/off).

Pour chaque combinaison, on calcule : moyenne, médiane, écart-type, min, max, et quartiles
(25%/75%). L'écart entre moyenne et médiane donne déjà une indication d'asymétrie de la distribution
(ex : si la moyenne est nettement au-dessus de la médiane, la distribution a une queue vers les
valeurs hautes — typique d'un CPU% qui reste bas la plupart du temps avec des pics de charge occasionnels).

In [12]:
# describe() calcule directement count/mean/std/min/25%/50%(médiane)/75%/max.
# groupby(...).describe() l'applique séparément à chaque combinaison node_name x workload_status.
# .T (transpose) pour un affichage plus lisible : une colonne par groupe, une ligne par statistique.
desc_stats = df_all.groupby(["node_name", "workload_status"], observed=True)[metric_cols].describe().T
desc_stats

node_name              rpi4                  vm-node             
workload_status         off          on          off           on
cpu_pct  count    8882.0000   8823.0000    8910.0000    8823.0000
         mean        0.5301      0.1193       0.2326       0.2079
         std         0.2049      0.0391       0.0426       0.0460
         min         0.0000      0.0000       0.1365       0.1260
         25%         0.5569      0.1010       0.2024       0.1733
         50%         0.6061      0.1086       0.2275       0.2018
         75%         0.6430      0.1290       0.2576       0.2343
         max         0.8722      0.7459       0.4725       0.5931
mem_pct  count    8882.0000   8823.0000    8910.0000    8823.0000
         mean        0.6621      0.7876       0.5877       0.5509
         std         0.1216      0.0210       0.0086       0.0138
         min         0.0000      0.7636       0.5695       0.5242
         25%         0.7078      0.7741       0.5812       0.5415
         50%         0.7142      0.7788       0.5864       0.5472
         75%         0.7193      0.7869       0.5928       0.5556
         max         0.7568      0.8492       0.6157       0.6397
fs_pct   count    8882.0000   8823.0000    8910.0000    8823.0000
         mean        0.7306      0.6919       0.7764       0.7604
         std         0.0163      0.0004       0.0027       0.0005
         min         0.0000      0.6870       0.7746       0.7597
         25%         0.7316      0.6916       0.7748       0.7600
         50%         0.7323      0.6919       0.7750       0.7602
         75%         0.7342      0.6922       0.7753       0.7605
         max         0.7377      0.6944       0.7827       0.7622
energy_w count    8882.0000   8823.0000    8910.0000    8823.0000
         mean       89.8674     83.6120      84.4825      84.3927
         std         7.3630      1.0504       0.6630       1.0772
         min         0.0000     82.1667      75.6159      47.8030
         25%        87.8696     83.0244      84.0843      83.8379
         50%        90.7434     83.2783      84.3587      84.1638
         75%        93.3907     84.2353      84.7395      84.8272
         max       113.8098     98.5163      94.8238     102.1222
rx_bps   count    8882.0000   8823.0000    8910.0000    8823.0000
         mean    89050.4883  37959.8944  931229.5782  698618.5791
         std     46592.7837  21780.8539  186079.0988  112383.8980
         min         0.0000      0.0000  572305.6333  473778.9667
         25%     65375.1333  20718.6833  816174.9417  625716.1000
         50%     92134.7000  37250.3667  894740.7000  684281.3333
         75%    121119.1250  40330.5500  993563.8083  754638.6667
         max    882718.6333 433919.4000 3750528.7365 1824759.0000
tx_bps   count    8882.0000   8823.0000    8910.0000    8823.0000
         mean   111675.8909  43098.0084 1491485.4059  908371.4465
         std     42511.5270  21308.7797  274220.3601  171368.8332
         min         0.0000      0.0000  964102.2000  623420.9000
         25%     87608.7500  25581.8167 1308646.7750  802536.1333
         50%    115134.3167  42772.0000 1434523.5000  896978.8333
         75%    142561.1083  45295.4667 1591368.0667  989185.9667
         max    474034.9765 436270.3000 6199251.7333 8209392.8000

### 10.1 Coefficient de variation (dispersion relative)

Le coefficient de variation (CV = écart-type / moyenne) mesure la dispersion **relative**
à l'échelle de la variable, contrairement à l'écart-type brut qui dépend de l'unité.
Ça permet de comparer la "volatilité" du rpi4 vs de la vm-node même si leurs moyennes
absolues de CPU/énergie sont très différentes. Un CV élevé = la mesure varie beaucoup
autour de sa moyenne ; un CV faible = la mesure est stable.

In [13]:
grouped = df_all.groupby(["node_name", "workload_status"], observed=True)[metric_cols]

mean_ = grouped.mean()
std_ = grouped.std()

# CV en %, pour une lecture plus intuitive (ex: 15% de dispersion relative autour de la moyenne).
cv = (std_ / mean_ * 100).round(2)
cv

cpu_pct  mem_pct  fs_pct  energy_w  rx_bps  tx_bps
node_name workload_status                                                    
rpi4      off              38.6500  18.3600  2.2200    8.1900 52.3200 38.0700
          on               32.7900   2.6700  0.0600    1.2600 57.3800 49.4400
vm-node   off              18.3100   1.4600  0.3500    0.7800 19.9800 18.3900
          on               22.1400   2.5100  0.0700    1.2800 16.0900 18.8700

### 10.2 Comparaison pods_on vs pods_off (delta de médiane)

On calcule directement l'écart de médiane entre la condition "on" (pods actifs) et "off" (idle),
par nœud. C'est une première estimation du "coût" du workload par rapport à la baseline —
une analyse par test statistique (ex: Mann-Whitney) pourra affiner ça plus tard, mais ici on
reste au niveau descriptif.

In [14]:
# Médiane par nœud et par condition (moins sensible aux valeurs extrêmes que la moyenne).
median_ = df_all.groupby(["node_name", "workload_status"], observed=True)[metric_cols].median()

# unstack("workload_status") : passe "on"/"off" de lignes à colonnes,
# ce qui permet de calculer la différence colonne à colonne facilement.
median_wide = median_.unstack("workload_status")

for col in metric_cols:
    delta = median_wide[col]["on"] - median_wide[col]["off"]
    print(f"{col:10s} — delta médiane (on - off) par nœud :")
    print(delta.to_string())
    print()

cpu_pct    — delta médiane (on - off) par nœud :
node_name
rpi4      -0.4975
vm-node   -0.0257

mem_pct    — delta médiane (on - off) par nœud :
node_name
rpi4       0.0646
vm-node   -0.0393

fs_pct     — delta médiane (on - off) par nœud :
node_name
rpi4      -0.0403
vm-node   -0.0148

energy_w   — delta médiane (on - off) par nœud :
node_name
rpi4      -7.4651
vm-node   -0.1949

rx_bps     — delta médiane (on - off) par nœud :
node_name
rpi4       -54884.3333
vm-node   -210459.3667

tx_bps     — delta médiane (on - off) par nœud :
node_name
rpi4       -72362.3167
vm-node   -537544.6667



### 10.3 Corrélations entre variables (par nœud)

On regarde comment les variables évoluent ensemble, séparément pour chaque nœud
(toujours pour la même raison : mélanger rpi4 et vm-node fausserait la corrélation,
puisqu'une partie de la corrélation viendrait juste du fait que ce sont deux
machines différentes, pas d'une vraie relation entre les variables).

On s'attend en particulier à une corrélation positive marquée entre `cpu_pct` et `energy_w`
(plus le CPU travaille, plus il consomme). La force de ce lien peut différer entre les deux
nœuds selon leur architecture matérielle.

In [15]:
for node in df_all["node_name"].cat.categories:
    subset = df_all[df_all["node_name"] == node]
    print(f"--- Corrélations ({node}) ---")
    # corr() calcule le coefficient de corrélation de Pearson entre chaque paire de colonnes.
    print(subset[metric_cols].corr().round(2))
    print()

--- Corrélations (rpi4) ---
          cpu_pct  mem_pct  fs_pct  energy_w  rx_bps  tx_bps
cpu_pct    1.0000  -0.0400  0.7900    0.6500  0.7500  0.8300
mem_pct   -0.0400   1.0000 -0.3400    0.0200  0.0600 -0.0500
fs_pct     0.7900  -0.3400  1.0000    0.4900  0.5800  0.7000
energy_w   0.6500   0.0200  0.4900    1.0000  0.5900  0.6400
rx_bps     0.7500   0.0600  0.5800    0.5900  1.0000  0.9500
tx_bps     0.8300  -0.0500  0.7000    0.6400  0.9500  1.0000

--- Corrélations (vm-node) ---
          cpu_pct  mem_pct  fs_pct  energy_w  rx_bps  tx_bps
cpu_pct    1.0000   0.2800  0.3100    0.2100  0.3400  0.3900
mem_pct    0.2800   1.0000  0.8900    0.1700  0.5800  0.7600
fs_pct     0.3100   0.8900  1.0000    0.1300  0.6500  0.8500
energy_w   0.2100   0.1700  0.1300    1.0000  0.2800  0.2900
rx_bps     0.3400   0.5800  0.6500    0.2800  1.0000  0.9000
tx_bps     0.3900   0.7600  0.8500    0.2900  0.9000  1.0000

